<a href="https://colab.research.google.com/github/Aryandore/Deep-Learning/blob/main/RNN_for_Sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data


In [2]:
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv("/content/drive/MyDrive/IMDB Dataset.csv")

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.isnull().sum()

,0
review,0
sentiment,0


In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape

(49582, 2)

# Pre-Processing


In [9]:
# 1 converting to lower case value
df['review'] = df['review'].str.lower()

In [10]:
# 2 removing URLs
import re
def remove_urls(text):
  text = re.sub(r'http\S+', '', text)
  return text
df['review'] = df['review'].apply(remove_urls)

In [11]:
# 3 Removing Punctuation
def remove_Punctuation(text):
  text = re.sub(r'[^A-Za-z0-9\s]', '', text)
  return text
df['review'] = df['review'].apply(remove_Punctuation)

In [12]:
# 4 removing HTML tags
def remove_html(text):
  text = re.sub(r'<.*?>', '', text)
  return text
df['review'] = df['review'].apply(remove_html)

In [13]:
# 5 removing StopWords
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = set(stopwords.words('english'))

  for word in tokens:
    if word in stop_words:
      text = text.replace(word,"")
  return text

df['review'] = df['review'].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [17]:
# 6 Stemming
# running -> run
from nltk.stem import PorterStemmer
def stemming(text):
  ps = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)
  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)
  return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)



In [18]:
# 7 encoding target variable
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])

In [19]:
y = df['sentiment']

In [20]:
# 8 Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df['review'])

## Dataset & Data Loaders


In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [22]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [23]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [24]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [25]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

## Build our RNN

In [28]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class RNN(torch.nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN Layer
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    # fully connected layer
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    # optional => shape(num of layers , batch size , hidden size)
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out,_ = self.rnn(x, h0)
    out = self.fc(out[:, -1, :])

    return out




In [46]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


## Training the RNN

In [47]:
epochs = 10
for epoch in range(epochs):
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()
    Xb = Xb.unsqueeze(1)
    output = model(Xb)
    output = torch.sigmoid_(output.squeeze())
    loss = criterion(output, yb)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch+1}/{epochs} Loss: {loss.item()}")

Epoch 1/10 Loss: 0.4303422272205353
Epoch 2/10 Loss: 0.1899963617324829
Epoch 3/10 Loss: 0.1565270572900772
Epoch 4/10 Loss: 0.23425331711769104
Epoch 5/10 Loss: 0.18763785064220428
Epoch 6/10 Loss: 0.13539856672286987
Epoch 7/10 Loss: 0.4129962921142578
Epoch 8/10 Loss: 0.3549189269542694
Epoch 9/10 Loss: 0.19970294833183289
Epoch 10/10 Loss: 0.1464412659406662


In [54]:
# evaluate

model.eval()
with torch.no_grad():
  correct = 0
  total = 0
  for Xb, yb in test_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)

    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
    total += yb.size(0)
    correct += (predicted == yb).sum().item()

  print(f"accuracy = {correct / total + 100}")

accuracy = 100.85761823131996
